![Databricks Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-end-to-end-en_us-1.0.0/images/20260922T145855Z/Get Started with Databricks End to End/Includes/images/databricks_academy_logo.png)

# Demo - Genie Agents

## Overview

Databricks Genie is an AI-powered conversational interface that lets business users and analysts query data using plain English. Genie translates natural language questions into SQL queries, executes them against your data, and returns results in an easy-to-interpret format.

In this demo, you explore a **Genie Agent** configured against a sample business dataset. You ask a series of business questions, observe how Genie interprets and responds to each one, and learn how to evaluate and refine those responses.

## Learning Objectives

By the end of this demonstration, you will be able to:

1. **Navigate** a Genie Agent and understand its components and configuration.
2. **Ask** business questions in natural language and observe how Genie translates them into SQL.
3. **Interpret** Genie responses, including result tables, charts, and generated SQL.
4. **Refine** follow-up questions to improve or narrow Genie's answers.
5. **Recognize** when Genie is the right tool, and when to reach for a SQL editor or dashboard instead.

## A. Sample Data

You use the `bakehouse` data in the `samples.bakehouse` catalog and schema.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

Databricks Free Edition has an example Genie Agent based on this data. You create a new one here in order to better understand the process.

  </div>
</div>

## B. Introduction to Genie


### B1. What is a Genie Agent?

A **Genie Agent** is a curated AI/BI environment in Databricks that connects a conversational AI interface to one or more Unity Catalog tables. A workspace admin or data steward configures the agent by:

- Selecting the tables Genie can query.
- Providing optional context (descriptions, business definitions, sample questions).
- Setting access permissions for end users.

Once configured, business users can open the Genie Agent and start asking questions. Genie handles the SQL generation, execution, and result presentation automatically.

The table below summarizes the key components of a Genie Agent:

| Component | Description |
|---|---|
| **Data Sources** | Unity Catalog tables selected by the agent admin |
| **Context Instructions** | Business definitions and rules that guide Genie's interpretation |
| **Sample Questions** | Pre-seeded questions that help users get started |
| **Conversation Thread** | The running history of questions and answers in a session |
| **Generated SQL** | The SQL Genie wrote to answer each question (viewable by users) |


## C. Navigating the Genie Interface


### C1. Create a Genie Agent

1. In the left navigation bar, right-click **Genie Agents** and select **Open Link in New Tab**.

2. Click **+ New** in the upper-right corner.

3. A pop-up appears prompting you to connect your data. In the pop-up, select **All** to locate the tables:

   - **Catalog**: `samples`
   - **Schema**: `bakehouse`
   - **Tables**:
     - `media_customer_reviews`
     - `media_gold_reviews_chunked`
     - `sales_customers`
     - `sales_franchises`
     - `sales_suppliers`
     - `sales_transactions`

4. When finished selecting data, click **Create** at the bottom.
    - The Genie Agent UI opens with the chat on the left and settings on the right. 

5. With the **Configure** button at the top selected, click **About**. 

6. Under **Description**, add the following: `This agent is designed to query the details of the bakehouse dataset.`

7. Under **Common questions**, add the AI generated common questions for users. These are sample questions appear when users open the Genie Agent. They help new users see what is possible.

8. Scroll down and find **About this agent**. Here you can edit the following:

    - **Title**: `YOUR_INITIALS - Basic Retail Details`
    - **Default warehouse**: Serverless Starter Warehouse (or any other available to you)
    - Click **Save** at the bottom to confirm the edits.

### C2. Key Interface Elements

Once inside a Genie Agent, you will see the following interface elements:

- **Question input bar**: Type your natural language question here and press Enter or click the send button.
- **Response area**: Genie displays its answer here, including result tables, charts, and a summary.
- **View SQL button**: Click this to see the exact SQL query Genie generated to produce the result.
- **Follow-up suggestions**: Genie may suggest related questions you can click to continue the conversation.
- **Conversation history**: Previous questions and answers in the current thread appear above the input bar.

Each conversation thread is independent. Starting a new thread clears the context from the previous session.


## D. Asking Business Questions in Natural Language


### D1. Ask Your First Question

Begin with a broad, high-level business question to see how Genie handles open-ended queries.

1. In the question bar, type:

   > *Show me the top 5 products by units sold.*

2. Observe the response. Genie will:

   - Generate and execute a SQL query against the `bakehouse` data.
   - Return the answer.
   - Optionally render a bar chart summarizing the data.

3. Click the dropdown arrow to see the results table, and click **Show code** in the response to inspect the query Genie wrote.

### D2. Interpret the SQL Genie Generated

Reviewing the generated SQL is a critical step in validating Genie's response. It tells you:

- Which tables were queried.
- How date filters were applied.
- Whether aggregations and groupings match your intent.

A typical generated query for the question above might look like this:

```sql
WITH product_units AS (
  SELECT
    `product`,
    SUM(`quantity`) AS total_units
  FROM
    `samples`.`bakehouse`.`sales_transactions`
  WHERE
    `product` IS NOT NULL
    AND `quantity` IS NOT NULL
  GROUP BY
    `product`
)
SELECT
  `product`,
  total_units
FROM
  (
    SELECT
      *,
      RANK() OVER (ORDER BY total_units DESC) AS rank
    FROM
      product_units
  ) ranked
WHERE
  rank <= 5;
```

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Validating Date Interpretation
  </strong>
  <div style="color:#333;">

Relative time expressions like "last quarter" or "this year" are interpreted by Genie based on the current date at query time. Always verify that the generated SQL date filters match your intended time window, especially when sharing results with stakeholders.

  </div>
</div>

### D3. Ask a More Specific Business Question

Now ask a more targeted question that requires joining multiple tables and applying a filter.

1. In the question bar, type:

   > *Show the total sales revenue that is greater than 2500 for each franchise, including the franchise name and the supplier name.*

2. Review the response table and any chart Genie renders.

3. Click **Show code** to confirm that Genie:

   - Joined `sales_transactions`, `sales_franchises`, and `sales_suppliers`.
   - Filtered by sales revenue greater than 2500.

If Genie's interpretation does not match your intent, you can correct it with a follow-up question, as shown next.

### D4. Interpret Ambiguous or Unexpected Responses

Not every question produces a perfect answer on the first attempt. Genie may:

- Give a first answer that needs clarification.
- Apply unexpected filters or groupings.
- Return an empty result due to overly restrictive conditions.

When this happens, use the following strategies:

1. **Review the generated SQL**: Identify where Genie's interpretation diverged from your intent.
2. **Rephrase the question**: Add specificity, for example "top 10 customers by total revenue in 2024".
3. **Provide clarifying context**: Type a correction such as *"By 'top customers' I mean those with the highest total revenue."*
4. **Check business definitions**: If a term is consistently misinterpreted, an agent admin can add a definition to the Genie Agent context instructions.


## E. When to Choose Genie Agents

AI/BI Dashboards and Genie Agents are complements and used for specific objectives. They answer different kinds of questions for end users.

<br></br>

<div class="vs-wrap">

<style>
.vs-wrap {
    width: 100%;
    display: flex;
    flex-direction: column;
    align-items: center;
}

.vs-row {
    display: flex;
    gap: 30px;
    justify-content: center;
    flex-wrap: wrap;
    max-width: 1100px;
    width: 100%;
}

.vs-card {
    flex: 1;
    min-width: 460px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 22px;
    position: relative;
    box-sizing: border-box;
    color: #0b2026;
}

.vs-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.vs-card.dash::before  { background: #FF5F46; }
.vs-card.genie::before { background: #FF5F46; }

.vs-icon {
    max-height: 48px;
    max-width: 100%;
    height: auto;
    width: auto;
    display: block;
    margin-bottom: 12px;
    object-fit: contain;
    image-rendering: -webkit-optimize-contrast;
}

.vs-head {
    font-weight: 700;
    font-size: 18pt;
    color: #0b2026;
    margin: 10px 0 14px 0;
}

.vs-body {
    font-size: 14pt;
    line-height: 1.5;
}

.vs-body h4 {
    margin: 0 0 8px 0;
    font-size: 15pt;
    color: #0b2026;
}

.vs-body ul {
    margin: 0 0 16px 0;
    padding-left: 18px;
}

.vs-body li { margin-bottom: 10px; }

.vs-footnote {
    max-width: 1100px;
    margin: 28px auto 0 auto;
    padding: 16px 24px;
    background: #F8F9FC;
    border: 3px solid #FF5F46;
    border-radius: 8px;
    text-align: center;
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.5;
}
</style>

<div class="vs-row">

<div class="vs-card dash">
<svg class="fam-wordmark" style="height:42px;width:auto;display:block;margin-bottom:4px;" viewBox="0 0 1104 200" fill="none" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Databricks AI/BI Dashboards">
<path d="M174.975 25.0001L107.787 25.0001L107.787 81.2499L174.975 81.2499L174.975 25.0001Z" fill="#FABFBA"/>
<path d="M107.812 175L174.999 175L174.999 93.7499L107.812 93.7499L107.812 175Z" fill="#FF5F46"/>
<path d="M92.1865 24.9999L24.9991 24.9999L24.999 118.75L92.1865 118.75L92.1865 24.9999Z" fill="#FF5F46"/>
<path d="M25.0254 175L92.2129 175L92.2129 132.031L25.0254 132.031L25.0254 175Z" fill="#FABFBA"/>
<path d="M234.76 137.5L263.022 60.22H277.816L306.078 137.5H292.058L270.309 75.676L248.56 137.5H234.76ZM247.346 119.284L250.878 108.796H288.966L292.499 119.284H247.346ZM313.397 137.5V60.22H326.645V137.5H313.397ZM333.682 149.423L361.392 51.94H374.309L346.599 149.423H333.682ZM387.093 137.5V60.22H417.343C422.789 60.22 427.315 61.0664 430.922 62.7592C434.602 64.452 437.362 66.8072 439.202 69.8248C441.042 72.7688 441.962 76.1912 441.962 80.092C441.962 83.9928 441.115 87.268 439.423 89.9176C437.73 92.5672 435.485 94.5912 432.688 95.9896C429.965 97.388 426.984 98.2344 423.746 98.5288L425.402 97.3144C428.861 97.388 431.952 98.308 434.675 100.074C437.472 101.841 439.68 104.159 441.299 107.03C442.919 109.826 443.728 112.954 443.728 116.414C443.728 120.462 442.735 124.105 440.747 127.343C438.834 130.508 436 133.01 432.247 134.85C428.493 136.617 423.893 137.5 418.447 137.5H387.093ZM400.341 126.681H416.459C420.802 126.681 424.187 125.65 426.616 123.59C429.045 121.529 430.259 118.658 430.259 114.978C430.259 111.298 429.008 108.391 426.506 106.257C424.003 104.049 420.581 102.945 416.239 102.945H400.341V126.681ZM400.341 92.8984H415.466C419.735 92.8984 422.973 91.9416 425.181 90.028C427.389 88.0408 428.493 85.3176 428.493 81.8584C428.493 78.4728 427.389 75.8232 425.181 73.9096C422.973 71.9224 419.698 70.9288 415.355 70.9288H400.341V92.8984ZM453.104 137.5V60.22H466.352V137.5H453.104ZM502.22 137.5V60.22H527.17C536.149 60.22 543.546 61.8024 549.36 64.9672C555.248 68.132 559.591 72.6216 562.388 78.436C565.258 84.1768 566.693 91.0216 566.693 98.9704C566.693 106.772 565.258 113.58 562.388 119.394C559.591 125.135 555.248 129.588 549.36 132.753C543.546 135.918 536.149 137.5 527.17 137.5H502.22ZM515.468 126.239H526.508C533.205 126.239 538.468 125.172 542.295 123.038C546.196 120.83 548.956 117.702 550.575 113.654C552.268 109.532 553.114 104.638 553.114 98.9704C553.114 93.2296 552.268 88.3352 550.575 84.2872C548.956 80.1656 546.196 77.0008 542.295 74.7928C538.468 72.5112 533.205 71.3704 526.508 71.3704H515.468V126.239ZM594.423 138.825C589.859 138.825 586.069 138.052 583.051 136.506C580.034 134.961 577.789 132.9 576.317 130.324C574.845 127.748 574.109 124.951 574.109 121.934C574.109 118.401 574.992 115.346 576.759 112.77C578.599 110.194 581.285 108.207 584.818 106.809C588.351 105.337 592.693 104.601 597.845 104.601H611.866C611.866 101.657 611.461 99.228 610.651 97.3144C609.842 95.3272 608.591 93.8552 606.898 92.8984C605.205 91.9416 603.034 91.4632 600.384 91.4632C597.367 91.4632 594.791 92.1624 592.656 93.5608C590.522 94.8856 589.197 96.9464 588.682 99.7432H575.655C576.096 95.7688 577.421 92.3832 579.629 89.5864C581.837 86.716 584.744 84.508 588.351 82.9624C592.031 81.3432 596.042 80.5336 600.384 80.5336C605.61 80.5336 610.063 81.4536 613.743 83.2936C617.423 85.06 620.219 87.636 622.133 91.0216C624.12 94.3336 625.114 98.3448 625.114 103.055V137.5H613.853L612.528 128.558C611.792 130.03 610.835 131.391 609.658 132.642C608.554 133.894 607.266 134.998 605.794 135.954C604.322 136.838 602.629 137.537 600.715 138.052C598.875 138.567 596.778 138.825 594.423 138.825ZM597.403 128.337C599.538 128.337 601.415 127.969 603.034 127.233C604.727 126.423 606.162 125.319 607.339 123.921C608.591 122.449 609.547 120.83 610.21 119.063C610.872 117.297 611.314 115.42 611.535 113.433V113.212H599.28C596.704 113.212 594.57 113.543 592.877 114.206C591.184 114.794 589.97 115.678 589.234 116.855C588.498 118.033 588.13 119.394 588.13 120.94C588.13 122.486 588.498 123.81 589.234 124.914C589.97 126.018 591.037 126.865 592.435 127.454C593.834 128.042 595.49 128.337 597.403 128.337ZM658.147 138.825C653.216 138.825 648.911 138.052 645.231 136.506C641.624 134.887 638.754 132.679 636.619 129.882C634.559 127.012 633.344 123.737 632.976 120.057H646.224C646.592 121.602 647.255 123.038 648.211 124.362C649.168 125.614 650.456 126.607 652.075 127.343C653.768 128.079 655.755 128.447 658.037 128.447C660.245 128.447 662.048 128.153 663.447 127.564C664.845 126.902 665.875 126.055 666.538 125.025C667.2 123.994 667.531 122.89 667.531 121.713C667.531 119.946 667.053 118.585 666.096 117.628C665.139 116.671 663.741 115.935 661.901 115.42C660.135 114.831 658 114.279 655.498 113.764C652.848 113.249 650.272 112.623 647.77 111.887C645.341 111.078 643.133 110.084 641.146 108.906C639.232 107.729 637.687 106.22 636.509 104.38C635.405 102.54 634.853 100.295 634.853 97.6456C634.853 94.4072 635.699 91.5 637.392 88.924C639.159 86.348 641.661 84.324 644.899 82.852C648.211 81.3064 652.149 80.5336 656.712 80.5336C663.189 80.5336 668.341 82.0424 672.168 85.06C676.069 88.004 678.351 92.1624 679.013 97.5352H666.427C666.059 95.4744 665.029 93.892 663.336 92.788C661.643 91.6104 659.399 91.0216 656.602 91.0216C653.731 91.0216 651.523 91.5736 649.978 92.6776C648.506 93.708 647.77 95.1064 647.77 96.8728C647.77 97.9768 648.211 99.0072 649.095 99.964C650.051 100.847 651.413 101.62 653.179 102.282C654.946 102.871 657.117 103.46 659.693 104.049C663.815 104.858 667.458 105.815 670.623 106.919C673.787 108.023 676.29 109.642 678.13 111.777C679.97 113.911 680.89 116.929 680.89 120.83C680.963 124.362 680.043 127.49 678.13 130.214C676.29 132.937 673.677 135.071 670.291 136.617C666.906 138.089 662.858 138.825 658.147 138.825ZM690.433 137.5V58.012H703.681V90.4696C705.521 87.452 708.023 85.06 711.188 83.2936C714.353 81.4536 717.922 80.5336 721.897 80.5336C726.46 80.5336 730.324 81.4904 733.489 83.404C736.653 85.244 739.045 88.004 740.665 91.684C742.357 95.2904 743.204 99.8168 743.204 105.263V137.5H730.066V106.478C730.066 101.694 729.036 98.0504 726.975 95.548C724.988 93.0456 721.933 91.7944 717.812 91.7944C715.162 91.7944 712.77 92.4568 710.636 93.7816C708.501 95.0328 706.809 96.8728 705.557 99.3016C704.306 101.657 703.681 104.564 703.681 108.023V137.5H690.433ZM785.76 138.825C782.816 138.825 780.166 138.42 777.811 137.61C775.456 136.874 773.395 135.844 771.628 134.519C769.862 133.121 768.39 131.538 767.212 129.772L765.777 137.5H753.964V58.012H767.212V90.028C768.979 87.3048 771.371 85.06 774.388 83.2936C777.406 81.4536 781.16 80.5336 785.649 80.5336C790.875 80.5336 795.475 81.8216 799.449 84.3976C803.497 86.9 806.662 90.3592 808.944 94.7752C811.299 99.1176 812.476 104.122 812.476 109.79C812.476 115.31 811.336 120.278 809.054 124.694C806.772 129.11 803.608 132.569 799.56 135.071C795.585 137.574 790.985 138.825 785.76 138.825ZM783 127.343C786.091 127.343 788.851 126.607 791.28 125.135C793.708 123.59 795.585 121.529 796.91 118.953C798.308 116.303 799.008 113.212 799.008 109.679C799.008 106.22 798.308 103.166 796.91 100.516C795.585 97.8664 793.708 95.8056 791.28 94.3336C788.851 92.8616 786.091 92.1256 783 92.1256C779.835 92.1256 777.038 92.8616 774.609 94.3336C772.254 95.8056 770.377 97.8664 768.979 100.516C767.654 103.166 766.992 106.22 766.992 109.679C766.992 113.212 767.654 116.303 768.979 118.953C770.377 121.529 772.254 123.59 774.609 125.135C777.038 126.607 779.835 127.343 783 127.343ZM847.968 138.825C842.669 138.825 837.885 137.61 833.616 135.182C829.421 132.679 826.109 129.257 823.68 124.914C821.325 120.498 820.147 115.457 820.147 109.79C820.147 103.975 821.362 98.8968 823.791 94.5544C826.219 90.1384 829.531 86.716 833.727 84.2872C837.995 81.7848 842.779 80.5336 848.079 80.5336C853.378 80.5336 858.125 81.7848 862.32 84.2872C866.589 86.716 869.901 90.1016 872.256 94.444C874.685 98.7864 875.899 103.865 875.899 109.679C875.899 115.494 874.685 120.572 872.256 124.914C869.827 129.257 866.479 132.679 862.21 135.182C858.015 137.61 853.267 138.825 847.968 138.825ZM847.968 127.454C850.618 127.454 853.01 126.791 855.144 125.466C857.352 124.142 859.119 122.154 860.443 119.505C861.768 116.855 862.431 113.58 862.431 109.679C862.431 105.778 861.768 102.54 860.443 99.964C859.192 97.3144 857.463 95.3272 855.255 94.0024C853.12 92.6776 850.728 92.0152 848.079 92.0152C845.503 92.0152 843.111 92.6776 840.903 94.0024C838.695 95.3272 836.928 97.3144 835.603 99.964C834.279 102.54 833.616 105.778 833.616 109.679C833.616 113.58 834.279 116.855 835.603 119.505C836.928 122.154 838.658 124.142 840.792 125.466C843 126.791 845.392 127.454 847.968 127.454ZM904.103 138.825C899.54 138.825 895.75 138.052 892.732 136.506C889.714 134.961 887.47 132.9 885.998 130.324C884.526 127.748 883.79 124.951 883.79 121.934C883.79 118.401 884.673 115.346 886.439 112.77C888.279 110.194 890.966 108.207 894.498 106.809C898.031 105.337 902.374 104.601 907.526 104.601H921.546C921.546 101.657 921.142 99.228 920.332 97.3144C919.522 95.3272 918.271 93.8552 916.578 92.8984C914.886 91.9416 912.714 91.4632 910.065 91.4632C907.047 91.4632 904.471 92.1624 902.337 93.5608C900.202 94.8856 898.878 96.9464 898.362 99.7432H885.335C885.777 95.7688 887.102 92.3832 889.31 89.5864C891.518 86.716 894.425 84.508 898.031 82.9624C901.711 81.3432 905.722 80.5336 910.065 80.5336C915.29 80.5336 919.743 81.4536 923.423 83.2936C927.103 85.06 929.9 87.636 931.814 91.0216C933.801 94.3336 934.794 98.3448 934.794 103.055V137.5H923.534L922.209 128.558C921.473 130.03 920.516 131.391 919.338 132.642C918.234 133.894 916.946 134.998 915.474 135.954C914.002 136.838 912.31 137.537 910.396 138.052C908.556 138.567 906.458 138.825 904.103 138.825ZM907.084 128.337C909.218 128.337 911.095 127.969 912.714 127.233C914.407 126.423 915.842 125.319 917.02 123.921C918.271 122.449 919.228 120.83 919.89 119.063C920.553 117.297 920.994 115.42 921.215 113.433V113.212H908.961C906.385 113.212 904.25 113.543 902.558 114.206C900.865 114.794 899.65 115.678 898.914 116.855C898.178 118.033 897.81 119.394 897.81 120.94C897.81 122.486 898.178 123.81 898.914 124.914C899.65 126.018 900.718 126.865 902.116 127.454C903.514 128.042 905.17 128.337 907.084 128.337ZM945.638 137.5V81.8584H957.451L958.665 92.236C959.99 89.8072 961.646 87.7464 963.633 86.0536C965.62 84.2872 967.939 82.9256 970.588 81.9688C973.311 81.012 976.292 80.5336 979.531 80.5336V94.5544H974.894C972.686 94.5544 970.588 94.8488 968.601 95.4376C966.614 95.9528 964.884 96.836 963.412 98.0872C962.014 99.2648 960.91 100.921 960.1 103.055C959.291 105.116 958.886 107.729 958.886 110.894V137.5H945.638ZM1009.75 138.825C1004.6 138.825 999.997 137.574 995.949 135.071C991.901 132.569 988.736 129.11 986.455 124.694C984.247 120.278 983.143 115.273 983.143 109.679C983.143 104.086 984.283 99.1176 986.565 94.7752C988.847 90.3592 992.011 86.9 996.059 84.3976C1000.18 81.8216 1004.82 80.5336 1009.97 80.5336C1014.17 80.5336 1017.81 81.3432 1020.9 82.9624C1024.06 84.508 1026.57 86.716 1028.41 89.5864V58.012H1041.65V137.5H1029.73L1028.41 129.441C1027.23 131.06 1025.79 132.606 1024.1 134.078C1022.41 135.476 1020.38 136.617 1018.03 137.5C1015.67 138.383 1012.91 138.825 1009.75 138.825ZM1012.51 127.343C1015.67 127.343 1018.47 126.607 1020.9 125.135C1023.33 123.59 1025.21 121.492 1026.53 118.842C1027.85 116.193 1028.52 113.138 1028.52 109.679C1028.52 106.22 1027.85 103.166 1026.53 100.516C1025.21 97.8664 1023.33 95.8056 1020.9 94.3336C1018.47 92.8616 1015.67 92.1256 1012.51 92.1256C1009.49 92.1256 1006.77 92.8616 1004.34 94.3336C1001.91 95.8056 999.997 97.8664 998.599 100.516C997.274 103.166 996.611 106.22 996.611 109.679C996.611 113.138 997.274 116.193 998.599 118.842C999.997 121.492 1001.91 123.59 1004.34 125.135C1006.77 126.607 1009.49 127.343 1012.51 127.343ZM1075.27 138.825C1070.34 138.825 1066.03 138.052 1062.35 136.506C1058.75 134.887 1055.88 132.679 1053.74 129.882C1051.68 127.012 1050.47 123.737 1050.1 120.057H1063.35C1063.71 121.602 1064.38 123.038 1065.33 124.362C1066.29 125.614 1067.58 126.607 1069.2 127.343C1070.89 128.079 1072.88 128.447 1075.16 128.447C1077.37 128.447 1079.17 128.153 1080.57 127.564C1081.97 126.902 1083 126.055 1083.66 125.025C1084.32 123.994 1084.65 122.89 1084.65 121.713C1084.65 119.946 1084.18 118.585 1083.22 117.628C1082.26 116.671 1080.86 115.935 1079.02 115.42C1077.26 114.831 1075.12 114.279 1072.62 113.764C1069.97 113.249 1067.39 112.623 1064.89 111.887C1062.46 111.078 1060.26 110.084 1058.27 108.906C1056.35 107.729 1054.81 106.22 1053.63 104.38C1052.53 102.54 1051.98 100.295 1051.98 97.6456C1051.98 94.4072 1052.82 91.5 1054.51 88.924C1056.28 86.348 1058.78 84.324 1062.02 82.852C1065.33 81.3064 1069.27 80.5336 1073.83 80.5336C1080.31 80.5336 1085.46 82.0424 1089.29 85.06C1093.19 88.004 1095.47 92.1624 1096.14 97.5352H1083.55C1083.18 95.4744 1082.15 93.892 1080.46 92.788C1078.77 91.6104 1076.52 91.0216 1073.72 91.0216C1070.85 91.0216 1068.65 91.5736 1067.1 92.6776C1065.63 93.708 1064.89 95.1064 1064.89 96.8728C1064.89 97.9768 1065.33 99.0072 1066.22 99.964C1067.17 100.847 1068.54 101.62 1070.3 102.282C1072.07 102.871 1074.24 103.46 1076.82 104.049C1080.94 104.858 1084.58 105.815 1087.74 106.919C1090.91 108.023 1093.41 109.642 1095.25 111.777C1097.09 113.911 1098.01 116.929 1098.01 120.83C1098.09 124.362 1097.17 127.49 1095.25 130.214C1093.41 132.937 1090.8 135.071 1087.41 136.617C1084.03 138.089 1079.98 138.825 1075.27 138.825Z" fill="#1B3139"/>
</svg>
<div class="vs-head">Use a Dashboard when...</div>
<div class="vs-body">
<h4>The question is known and repeated.</h4>
<ul>
<li>You have a <strong>recurring audience</strong> (weekly exec review, monthly sales meeting).</li>
<li>You want a <strong>curated, polished layout</strong> with consistent KPIs each time.</li>
<li>You want to control <strong>how the data is presented</strong>, not just what is returned.</li>
<li>Consumers should <strong>filter and drill</strong> but not write SQL.</li>
</ul>
</div>
</div>

<div class="vs-card genie">
<svg class="fam-wordmark" style="height:42px;width:auto;display:block;margin-bottom:4px;" viewBox="0 0 881 200" fill="none" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Databricks Genie Agents">
<path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/>
<path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/>
<path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/>
<path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/>
<rect x="572" y="58" width="308.12" height="84" rx="42" fill="#0B2026"/>
<path d="M672.51 113.74L682.313 86.9347H687.444L697.247 113.74H692.384L684.84 92.2958L677.296 113.74H672.51ZM676.875 107.422L678.1 103.784H691.312L692.537 107.422H676.875ZM715.676 114.2C713.149 114.2 710.928 113.625 709.013 112.477C707.124 111.328 705.656 109.732 704.609 107.69C703.563 105.622 703.039 103.209 703.039 100.452C703.039 97.7207 703.575 95.3082 704.648 93.2148C705.72 91.0959 707.252 89.4493 709.243 88.275C711.26 87.0751 713.647 86.4752 716.404 86.4752C719.518 86.4752 722.097 87.2283 724.139 88.7345C726.207 90.2152 727.547 92.283 728.16 94.9381H722.99C722.607 93.6105 721.854 92.5766 720.731 91.8363C719.608 91.0959 718.152 90.7258 716.365 90.7258C714.553 90.7258 712.996 91.1215 711.694 91.9129C710.417 92.6787 709.434 93.802 708.745 95.2827C708.081 96.7379 707.749 98.4738 707.749 100.491C707.749 102.507 708.081 104.243 708.745 105.699C709.434 107.154 710.404 108.264 711.655 109.03C712.906 109.77 714.374 110.141 716.059 110.141C718.561 110.141 720.45 109.439 721.727 108.034C723.003 106.63 723.756 104.805 723.986 102.558H717.055V99.0738H728.619V113.74H724.407L724.024 110.064C723.437 110.932 722.76 111.685 721.995 112.323C721.229 112.936 720.322 113.408 719.276 113.74C718.255 114.047 717.055 114.2 715.676 114.2ZM738.172 113.74V86.9347H755.481V90.6492H742.767V98.3462H754.332V101.984H742.767V110.026H755.481V113.74H738.172ZM764.906 113.74V86.9347H769.501L782.406 106.311V86.9347H787.001V113.74H782.406L769.501 94.4019V113.74H764.906ZM803.229 113.74V90.6492H795.341V86.9347H815.675V90.6492H807.786V113.74H803.229ZM832.683 114.2C830.742 114.2 829.019 113.855 827.513 113.166C826.007 112.477 824.832 111.494 823.99 110.217C823.148 108.941 822.714 107.422 822.688 105.66H827.551C827.551 106.554 827.756 107.358 828.164 108.073C828.598 108.762 829.185 109.311 829.925 109.719C830.691 110.128 831.61 110.332 832.683 110.332C833.602 110.332 834.393 110.192 835.057 109.911C835.746 109.605 836.269 109.183 836.627 108.647C837.01 108.086 837.201 107.435 837.201 106.694C837.201 105.852 836.997 105.162 836.589 104.626C836.206 104.065 835.67 103.592 834.98 103.209C834.291 102.827 833.5 102.495 832.606 102.214C831.713 101.907 830.768 101.588 829.772 101.257C827.653 100.542 826.058 99.6354 824.986 98.5377C823.913 97.4144 823.377 95.9209 823.377 94.0573C823.377 92.5 823.747 91.1597 824.488 90.0365C825.228 88.9132 826.262 88.0452 827.59 87.4325C828.943 86.7943 830.5 86.4752 832.261 86.4752C834.048 86.4752 835.606 86.7943 836.933 87.4325C838.286 88.0707 839.346 88.9642 840.112 90.1131C840.903 91.2363 841.311 92.5894 841.337 94.1722H836.435C836.41 93.5084 836.231 92.8957 835.899 92.3341C835.567 91.7469 835.082 91.2746 834.444 90.9172C833.831 90.5343 833.078 90.3428 832.185 90.3428C831.419 90.3173 830.73 90.4449 830.117 90.7258C829.53 90.981 829.058 91.364 828.7 91.8746C828.368 92.3596 828.202 92.9723 828.202 93.7127C828.202 94.4275 828.355 95.0274 828.662 95.5125C828.994 95.972 829.466 96.3677 830.079 96.6996C830.691 97.0059 831.406 97.2995 832.223 97.5803C833.04 97.8611 833.934 98.1675 834.904 98.4994C836.231 98.9334 837.431 99.4695 838.503 100.108C839.601 100.72 840.469 101.525 841.107 102.52C841.745 103.516 842.065 104.805 842.065 106.388C842.065 107.766 841.707 109.056 840.992 110.256C840.278 111.43 839.231 112.387 837.852 113.128C836.474 113.842 834.75 114.2 832.683 114.2Z" fill="white"/>
<path d="M632.893 74.0239C632.893 74.0239 632.893 74.02 632.893 74.0221L634.136 80.6146C635.485 87.7644 641.078 93.3568 648.228 94.7065L654.82 95.9495C654.82 97.6494 654.82 102.901 654.82 102.901L648.228 104.144C641.078 105.494 635.485 111.086 634.136 118.236L632.893 124.829C631.193 124.829 625.941 124.829 625.941 124.829L624.698 118.236C623.348 111.086 617.756 105.494 610.606 104.144L604.014 102.901C604.014 102.901 604.015 97.6494 604.015 95.9495H604.014L610.606 94.7065C617.756 93.3568 623.348 87.7644 624.698 80.6146L625.941 74.0221C625.941 74.0221 631.193 74.0239 632.893 74.0239Z" fill="#FF3621"/>
</svg>
<div class="vs-head">Use Genie Agents when...</div>
<div class="vs-body">
<h4>The question is unknown or one-off.</h4>
<ul>
<li>A business user has a <strong>follow-up question</strong> the dashboard does not answer.</li>
<li>You want <strong>self-service exploration</strong> for people who do not know SQL.</li>
<li>The question is <strong>ad-hoc or investigative</strong>, not a fixed report.</li>
<li>You need <strong>quick answers</strong> without building a new chart.</li>
</ul>
</div>
</div>

</div>

<div class="vs-footnote">
<strong>Better Together! Most organizations use both.</strong> Dashboards answer the questions you already know to ask. Genie helps uncover the ones you didn't.
</div>

</div>


### E1. Genie Agent Use Case Overview

<br></br>

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="display: flex; flex-direction: column; gap: 16px;">

<!-- Header -->
<div style="background: #1B5162; color: white; border-radius: 8px 8px 4px 4px; padding: 20px 24px; text-align: center;">
  <div style="font-size: 20pt; font-weight: 700;">Strong Use Cases for Genie Agents</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">A Genie Agent is particularly effective for the following use cases</div>
</div>

<!-- Row 1 -->
<div style="display: flex; gap: 12px;">

<div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #98182A; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 15pt; font-weight: 700; color: #0b2026; margin-bottom: 8px;">1. Self-service analytics</div>
  <div style="font-size: 14pt; color: #333; line-height: 1.5;">
    Business users can explore data without writing SQL or relying on a data team.
  </div>
</div>

<div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 15pt; font-weight: 700; color: #0b2026; margin-bottom: 8px;">2. Rapid prototyping</div>
  <div style="font-size: 14pt; color: #333; line-height: 1.5;">
    Analysts can quickly validate hypotheses before building formal dashboards.
  </div>
</div>

</div>

<!-- Row 2 -->
<div style="display: flex; gap: 12px;">

<div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #02A36F; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 15pt; font-weight: 700; color: #0b2026; margin-bottom: 8px;">3. Exploratory data analysis</div>
  <div style="font-size: 14pt; color: #333; line-height: 1.5;">
    Genie handles aggregations, filters, joins, and sorting with minimal guidance.
  </div>
</div>

<div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #FFAB00; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 15pt; font-weight: 700; color: #0b2026; margin-bottom: 8px;">4. Accessible reporting</div>
  <div style="font-size: 14pt; color: #333; line-height: 1.5;">
    Non-technical stakeholders can get answers to routine business questions on demand.
  </div>
</div>

</div>

</div>

<!-- Key point -->
<div style="margin-top: 20px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">The common thread:</div>
    Genie Agents shine when the goal is fast, self-directed answers in natural language, and the audience does not want to write SQL or wait on a report.
  </div>
</div>

</div>


### E2. Choosing Other Tools

Understanding what Genie does well helps you set appropriate expectations and design better Genie Agents. It also helps you decide when other tools might be more appropriate.

| Watch out for | Description |
|---|---|
| **Ambiguous terminology** | Genie may interpret business terms differently than intended without explicit context instructions |
| **Complex multi-step logic** | Questions requiring procedural logic or window functions may produce incorrect SQL |
| **Data modification** | Genie queries data. To modify data, use a SQL editor or notebook |
| **Context window** | For long analyses, start a new conversation thread for each major topic to keep Genie focused |
| **Schema dependency** | Genie's accuracy depends on well-named columns and tables with good descriptions |

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Improving Genie Agent Accuracy
  </strong>
  <div style="color:#333;">

Agent admins can significantly improve Genie's accuracy by adding column-level descriptions in Unity Catalog, providing business glossary definitions in the agent context, and seeding the agent with representative sample questions. 

See the <strong>Genie Agents</strong> documentation for configuration best practices: <a href="https://docs.databricks.com/aws/en/genie-agents/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/genie-agents/" target="_blank">GCP</a>.

  </div>
</div>

## Conclusion

In this demo, you:

1. Created a Genie Agent on the `samples.bakehouse` data and configured its settings.
2. Asked business questions in natural language and reviewed the SQL Genie generated.
3. Refined questions and used follow-ups to improve Genie's answers.
4. Identified when Genie is the right tool, and when to reach for a SQL editor or dashboard.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>